In [3]:
kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "learning-events")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_stream_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
from pyspark.sql.functions import (
    col,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    DoubleType
)


learning_event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("session_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("topic_id", StringType(), True),
    StructField("question_id", StringType(), True),
    StructField("attempt_number", IntegerType(), True),
    StructField("is_correct", BooleanType(), True),
    StructField("score", DoubleType(), True),
    StructField("hints_used", IntegerType(), True),
    StructField("attempt_duration_seconds", IntegerType(), True),
    StructField("event_time", StringType(), False),
    StructField("produced_at", StringType(), True),
])


parsed_stream_df = (
    kafka_stream_df
    .select(
        col("key").cast("string").alias("kafka_key"),
        col("value").cast("string").alias("raw_json"),
        col("topic").alias("kafka_topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .withColumn(
        "event",
        from_json(
            col("raw_json"),
            learning_event_schema
        )
    )
    .select(
        "event.*",
        "kafka_key",
        "kafka_topic",
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "raw_json"
    )
)

parsed_stream_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- topic_id: string (nullable = true)
 |-- question_id: string (nullable = true)
 |-- attempt_number: integer (nullable = true)
 |-- is_correct: boolean (nullable = true)
 |-- score: double (nullable = true)
 |-- hints_used: integer (nullable = true)
 |-- attempt_duration_seconds: integer (nullable = true)
 |-- event_time: string (nullable = true)
 |-- produced_at: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- raw_json: string (nullable = true)



In [5]:
spark.table(
    "demo.bronze.learning_events"
).printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- raw_payload: string (nullable = true)



In [6]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    to_timestamp
)

bronze_stream_df = (
    parsed_stream_df
    .filter(col("event_id").isNotNull())
    .select(
        col("event_id"),
        col("user_id"),
        col("session_id"),
        col("event_type"),

        to_timestamp(
            col("event_time")
        ).alias("event_time"),

        current_timestamp()
        .alias("ingestion_time"),

        lit("kafka")
        .alias("source_system"),

        col("raw_json")
        .alias("raw_payload")
    )
)

bronze_stream_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- source_system: string (nullable = false)
 |-- raw_payload: string (nullable = true)



In [7]:
checkpoint_path = (
    "/home/iceberg/notebooks/notebooks/checkpoints/"
    "learning_events_kafka_to_bronze"
)

streaming_query = (
    bronze_stream_df.writeStream
    .foreachBatch(
        lambda batch_df, batch_id: (
            batch_df
            .dropDuplicates(["event_id"])
            .writeTo("demo.bronze.learning_events")
            .append()
        )
    )
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

streaming_query.awaitTermination()

print("Kafka to Bronze streaming batch completed.")

26/07/29 09:31:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/29 09:31:27 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

Kafka to Bronze streaming batch completed.
Batch 0: merged 5 records
Accepted batch 0: merged successfully


In [8]:
spark.sql("""
SELECT
    event_id,
    user_id,
    session_id,
    event_type,
    event_time,
    ingestion_time,
    source_system
FROM demo.bronze.learning_events
WHERE source_system = 'kafka'
ORDER BY ingestion_time
""").show(truncate=False)

+--------+-------+----------+----------+----------+--------------+-------------+
|event_id|user_id|session_id|event_type|event_time|ingestion_time|source_system|
+--------+-------+----------+----------+----------+--------------+-------------+
+--------+-------+----------+----------+----------+--------------+-------------+



In [9]:
streaming_query = (
    bronze_stream_df.writeStream
    .foreachBatch(
        lambda batch_df, batch_id: (
            batch_df
            .dropDuplicates(["event_id"])
            .writeTo("demo.bronze.learning_events")
            .append()
        )
    )
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

streaming_query.awaitTermination()

print("Kafka to Bronze streaming batch completed.")

26/07/29 09:31:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/29 09:31:39 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Kafka to Bronze streaming batch completed.


In [10]:
spark.sql("""
SELECT
    event_id,
    user_id,
    session_id,
    event_type,
    event_time,
    ingestion_time,
    source_system
FROM demo.bronze.learning_events
WHERE source_system = 'kafka'
ORDER BY ingestion_time
""").show(truncate=False)

+--------+-------+----------+----------+----------+--------------+-------------+
|event_id|user_id|session_id|event_type|event_time|ingestion_time|source_system|
+--------+-------+----------+----------+----------+--------------+-------------+
+--------+-------+----------+----------+----------+--------------+-------------+



In [11]:
import os
import shutil

checkpoint_path = (
    "/home/iceberg/notebooks/notebooks/checkpoints/"
    "learning_events_kafka_to_bronze"
)

if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("Old Kafka checkpoint deleted.")
else:
    print("Checkpoint does not exist.")

Old Kafka checkpoint deleted.


In [12]:
streaming_query = (
    bronze_stream_df.writeStream
    .foreachBatch(
        lambda batch_df, batch_id: (
            batch_df
            .dropDuplicates(["event_id"])
            .writeTo("demo.bronze.learning_events")
            .append()
        )
    )
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

streaming_query.awaitTermination()

print("Kafka to Bronze streaming batch completed.")

26/07/29 09:34:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/29 09:34:02 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Kafka to Bronze streaming batch completed.


In [13]:
spark.sql("""
SELECT
    event_id,
    user_id,
    session_id,
    event_type,
    event_time,
    ingestion_time,
    source_system
FROM demo.bronze.learning_events
WHERE source_system = 'kafka'
ORDER BY ingestion_time
""").show(truncate=False)

+-------------------+-------+------------------+----------------+--------------------------+-----------------------+-------------+
|event_id           |user_id|session_id        |event_type      |event_time                |ingestion_time         |source_system|
+-------------------+-------+------------------+----------------+--------------------------+-----------------------+-------------+
|stream_629cc0b76c98|user_2 |stream_session_004|practice_attempt|2026-07-29 09:27:12.151975|2026-07-29 09:27:34.438|kafka        |
|stream_130a5b808cc9|user_1 |stream_session_003|practice_attempt|2026-07-29 09:27:11.137321|2026-07-29 09:27:34.438|kafka        |
|stream_d5d6f5cf054f|user_2 |stream_session_001|practice_attempt|2026-07-29 09:27:08.956975|2026-07-29 09:27:34.438|kafka        |
|stream_4f8d229effa9|user_3 |stream_session_005|practice_attempt|2026-07-29 09:27:13.165739|2026-07-29 09:27:34.438|kafka        |
|stream_eb268284ea65|user_3 |stream_session_002|practice_attempt|2026-07-29 09:27:1

In [14]:
spark.sql("""
DELETE FROM demo.bronze.learning_events
WHERE source_system = 'kafka'
""")

print("Old Kafka test rows deleted.")

Old Kafka test rows deleted.


In [15]:
import os
import shutil

checkpoint_path = (
    "/home/iceberg/notebooks/notebooks/checkpoints/"
    "learning_events_kafka_to_bronze"
)

if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)

print("Checkpoint reset for the clean MVP run.")

Checkpoint reset for the clean MVP run.


In [19]:
def merge_kafka_batch(batch_df, batch_id):
    clean_batch_df = (
        batch_df
        .filter("event_id IS NOT NULL")
        .dropDuplicates(["event_id"])
    )

    # אין מה לעבד ב-micro-batch ריק
    if clean_batch_df.isEmpty():
        print(f"Batch {batch_id}: no records")
        return

    batch_spark = batch_df.sparkSession
    view_name = f"current_kafka_learning_events_{batch_id}"

    clean_batch_df.createOrReplaceTempView(view_name)

    try:
        batch_spark.sql(f"""
            MERGE INTO demo.bronze.learning_events AS target
            USING {view_name} AS source

            ON target.event_id = source.event_id

            WHEN NOT MATCHED THEN INSERT (
                event_id,
                user_id,
                session_id,
                event_type,
                event_time,
                ingestion_time,
                source_system,
                raw_payload
            )
            VALUES (
                source.event_id,
                source.user_id,
                source.session_id,
                source.event_type,
                source.event_time,
                source.ingestion_time,
                source.source_system,
                source.raw_payload
            )
        """)

        print(
            f"Batch {batch_id}: merged "
            f"{clean_batch_df.count()} records"
        )

    finally:
        batch_spark.catalog.dropTempView(view_name)


streaming_query = (
    bronze_stream_df.writeStream
    .foreachBatch(merge_kafka_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

streaming_query.awaitTermination()

print("Kafka to Bronze streaming batch completed with MERGE.")

26/07/29 09:38:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/29 09:38:29 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Kafka to Bronze streaming batch completed with MERGE.


In [20]:
spark.sql("""
SELECT
    event_id,
    user_id,
    session_id,
    event_type,
    event_time,
    ingestion_time,
    source_system
FROM demo.bronze.learning_events
WHERE source_system = 'kafka'
ORDER BY event_time
""").show(truncate=False)

+-------------------+-------+------------------+----------------+--------------------------+-----------------------+-------------+
|event_id           |user_id|session_id        |event_type      |event_time                |ingestion_time         |source_system|
+-------------------+-------+------------------+----------------+--------------------------+-----------------------+-------------+
|stream_d5d6f5cf054f|user_2 |stream_session_001|practice_attempt|2026-07-29 09:27:08.956975|2026-07-29 09:36:20.773|kafka        |
|stream_eb268284ea65|user_3 |stream_session_002|practice_attempt|2026-07-29 09:27:10.116324|2026-07-29 09:36:20.773|kafka        |
|stream_130a5b808cc9|user_1 |stream_session_003|practice_attempt|2026-07-29 09:27:11.137321|2026-07-29 09:36:20.773|kafka        |
|stream_629cc0b76c98|user_2 |stream_session_004|practice_attempt|2026-07-29 09:27:12.151975|2026-07-29 09:36:20.773|kafka        |
|stream_4f8d229effa9|user_3 |stream_session_005|practice_attempt|2026-07-29 09:27:1

In [21]:
from pyspark.sql.functions import (
    col,
    expr,
    when
)

late_arrival_checked_df = (
    bronze_stream_df
    .withColumn(
        "arrival_delay_hours",
        (
            col("ingestion_time").cast("long")
            - col("event_time").cast("long")
        ) / 3600.0
    )
    .withColumn(
        "late_arrival_status",
        when(
            col("arrival_delay_hours") <= 48,
            "ACCEPTED"
        ).otherwise("TOO_LATE")
    )
)

late_arrival_checked_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- source_system: string (nullable = false)
 |-- raw_payload: string (nullable = true)
 |-- arrival_delay_hours: double (nullable = true)
 |-- late_arrival_status: string (nullable = false)



In [22]:
accepted_events_df = (
    late_arrival_checked_df
    .filter(col("late_arrival_status") == "ACCEPTED")
    .drop(
        "arrival_delay_hours",
        "late_arrival_status"
    )
)

too_late_events_df = (
    late_arrival_checked_df
    .filter(col("late_arrival_status") == "TOO_LATE")
)

In [23]:
from pyspark.sql.functions import col


def merge_accepted_batch(batch_df, batch_id):
    clean_batch_df = (
        batch_df
        .filter(col("event_id").isNotNull())
        .dropDuplicates(["event_id"])
    )

    if clean_batch_df.isEmpty():
        print(f"Accepted batch {batch_id}: no records")
        return

    batch_spark = batch_df.sparkSession
    view_name = f"accepted_kafka_events_{batch_id}"

    clean_batch_df.createOrReplaceTempView(view_name)

    try:
        batch_spark.sql(f"""
            MERGE INTO demo.bronze.learning_events AS target
            USING {view_name} AS source

            ON target.event_id = source.event_id

            WHEN NOT MATCHED THEN INSERT (
                event_id,
                user_id,
                session_id,
                event_type,
                event_time,
                ingestion_time,
                source_system,
                raw_payload
            )
            VALUES (
                source.event_id,
                source.user_id,
                source.session_id,
                source.event_type,
                source.event_time,
                source.ingestion_time,
                source.source_system,
                source.raw_payload
            )
        """)

        print(f"Accepted batch {batch_id}: merged successfully")

    finally:
        batch_spark.catalog.dropTempView(view_name)


accepted_query = (
    accepted_events_df.writeStream
    .foreachBatch(merge_accepted_batch)
    .option(
        "checkpointLocation",
        "/home/iceberg/notebooks/notebooks/checkpoints/"
        "accepted_learning_events"
    )
    .trigger(availableNow=True)
    .start()
)

accepted_query.awaitTermination()

print("Accepted events processing completed.")

26/07/29 09:50:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/29 09:50:06 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

Accepted events processing completed.
